## Step 1: Build the Initial Private Knowledge Base

In [2]:
import requests
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD
import math

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

EX = Namespace("http://example.org/")

# We choose different big football clubs to have more entities
CLUBS = {
    "Manchester United": "Q18656",
    "FC Barcelona":      "Q7156",
    "Real Madrid":       "Q8682",
    "Liverpool":         "Q1130849",
    "Bayern Munich":     "Q15789",
    "Paris Saint-Germain": "Q583",
    "Juventus":            "Q17050",
    "Manchester City":     "Q50602",
}

def fetch_players_for_club(club_qid: str, club_name: str) -> list:
    query = f"""
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>

    SELECT ?player ?playerLabel
           ?pos ?posLabel
           ?dob
           ?country ?countryLabel
           ?height
           ?birthPlace ?birthPlaceLabel
    WHERE {{
      ?player wdt:P31 wd:Q5 ;
              wdt:P106 wd:Q937857 ;
              wdt:P54 wd:{club_qid} .

      OPTIONAL {{ ?player wdt:P413 ?pos . }}
      OPTIONAL {{ ?player wdt:P569 ?dob . }}
      OPTIONAL {{ ?player wdt:P27 ?country . }}
      OPTIONAL {{ ?player wdt:P2048 ?height . }}
      OPTIONAL {{ ?player wdt:P19 ?birthPlace . }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
    }}
    """
    resp = requests.get(
        SPARQL_ENDPOINT,
        params={"query": query, "format": "json"},
        headers={"User-Agent": "FootballKB/1.0"}
    )
    results = resp.json()["results"]["bindings"]
    print(f"  {club_name}: {len(results)} joueurs")
    return results

# Collect for all clubs
all_results = []
for club_name, club_qid in CLUBS.items():
    rows = fetch_players_for_club(club_qid, club_name)
    # We add the club if origin in each row
    for r in rows:
        r["_club_qid"]  = club_qid
        r["_club_name"] = club_name
    all_results.extend(rows)

print(f"\nTotal lines collected : {len(all_results)}")

  Manchester United: 1740 joueurs
  FC Barcelona: 1526 joueurs
  Real Madrid: 926 joueurs
  Liverpool: 1547 joueurs
  Bayern Munich: 901 joueurs
  Paris Saint-Germain: 0 joueurs
  Juventus: 0 joueurs
  Manchester City: 1511 joueurs

Total lines collected : 8151


In [3]:
# IGNORE
def qid_from_wd_uri(uri):
    if not uri or not isinstance(uri, str):
        return None
    if "/entity/" in uri:
        return uri.rsplit("/entity/", 1)[1]
    return None

def to_xsd_date(value):
    if value is None or not isinstance(value, str):
        return None
    if "T" in value:
        value = value.split("T", 1)[0]
    if len(value) == 10 and value[4] == "-" and value[7] == "-":
        y, m, d = value[0:4], value[5:7], value[8:10]
        if y.isdigit() and m.isdigit() and d.isdigit():
            if 1 <= int(m) <= 12 and 1 <= int(d) <= 31:
                return value
    return None

g = Graph()
g.bind("ex",   EX)
g.bind("wd",   Namespace("http://www.wikidata.org/entity/"))
g.bind("wdt",  Namespace("http://www.wikidata.org/prop/direct/"))
g.bind("rdfs", RDFS)
g.bind("xsd",  XSD)

seen_players = set()

for r in all_results:
    # Player
    raw_player = r["player"]["value"]
    qid = qid_from_wd_uri(raw_player)
    if not qid or qid in seen_players:
        continue
    seen_players.add(qid)

    player_uri = EX[f"player/{qid}"]
    g.add((player_uri, RDF.type,    EX.FootballPlayer))
    g.add((player_uri, RDFS.label,  Literal(r.get("playerLabel", {}).get("value", qid), lang="en")))

    # Club
    club_qid  = r["_club_qid"]
    club_name = r["_club_name"]
    club_uri  = EX[f"club/{club_qid}"]
    g.add((club_uri,   RDF.type,   EX.Club))
    g.add((club_uri,   RDFS.label, Literal(club_name, lang="en")))
    g.add((player_uri, EX.playsFor, club_uri))

    # Position
    if "pos" in r:
        pos_qid = qid_from_wd_uri(r["pos"]["value"])
        if pos_qid:
            pos_uri = EX[f"position/{pos_qid}"]
            g.add((pos_uri,    RDF.type,   EX.Position))
            g.add((pos_uri,    RDFS.label, Literal(r.get("posLabel", {}).get("value", pos_qid), lang="en")))
            g.add((player_uri, EX.hasPosition, pos_uri))

    # Nationality
    if "country" in r:
        c_qid = qid_from_wd_uri(r["country"]["value"])
        if c_qid:
            c_uri = EX[f"country/{c_qid}"]
            g.add((c_uri,      RDF.type,   EX.Country))
            g.add((c_uri,      RDFS.label, Literal(r.get("countryLabel", {}).get("value", c_qid), lang="en")))
            g.add((player_uri, EX.hasNationality, c_uri))

    # Date of birth
    dob = to_xsd_date(r.get("dob", {}).get("value"))
    if dob:
        g.add((player_uri, EX.birthDate, Literal(dob, datatype=XSD.date)))

    # Height
    if "height" in r:
        try:
            h = float(r["height"]["value"])
            if not math.isnan(h) and 1.0 < h < 2.5:
                g.add((player_uri, EX.heightM, Literal(round(h, 2), datatype=XSD.decimal)))
        except:
            pass

    # Birth place
    if "birthPlace" in r:
        bp_qid = qid_from_wd_uri(r["birthPlace"]["value"])
        if bp_qid:
            bp_uri = EX[f"place/{bp_qid}"]
            g.add((bp_uri,     RDF.type,   EX.Place))
            g.add((bp_uri,     RDFS.label, Literal(r.get("birthPlaceLabel", {}).get("value", bp_qid), lang="en")))
            g.add((player_uri, EX.birthPlace, bp_uri))

g.serialize(destination="private_kb.ttl", format="turtle")
print("Saved: private_kb.ttl")
print("Triples:", len(g))

Saved: private_kb.ttl
Triples: 46103


In [4]:
import re


def qid_from_wd_uri(uri):
    if not uri or not isinstance(uri, str):
        return None
    if "/entity/" in uri:
        return uri.rsplit("/entity/", 1)[1]
    return None

def to_xsd_date(value):
    if value is None or not isinstance(value, str):
        return None
    if "T" in value:
        value = value.split("T", 1)[0]
    if len(value) == 10 and value[4] == "-" and value[7] == "-":
        y, m, d = value[0:4], value[5:7], value[8:10]
        if y.isdigit() and m.isdigit() and d.isdigit():
            if 1 <= int(m) <= 12 and 1 <= int(d) <= 31:
                return value
    return None
def make_uri(label: str) -> URIRef:
    """Convert a label to a clean camelCase URI like ex:LionelMessi"""
    if not label or not isinstance(label, str):
        return None
    clean = re.sub(r"[^a-zA-Z0-9 ]", "", label)
    camel = "".join(word.capitalize() for word in clean.split())
    if not camel:
        return None
    return EX[camel]

g = Graph()
g.bind("ex",   EX)
g.bind("rdfs", RDFS)
g.bind("xsd",  XSD)

# Maps ex:LionelMessi -> Q615 — needed for Step 2 linking
label_to_qid = {}

seen_players = set()

for r in all_results:
    raw_player   = r["player"]["value"]
    player_label = r.get("playerLabel", {}).get("value", "")
    qid          = qid_from_wd_uri(raw_player)

    if not qid or not player_label or qid in seen_players:
        continue
    seen_players.add(qid)

    player_uri = make_uri(player_label)
    if player_uri is None:
        continue

    g.add((player_uri, RDF.type,   EX.FootballPlayer))
    g.add((player_uri, RDFS.label, Literal(player_label, lang="en")))
    label_to_qid[str(player_uri)] = qid

    # Club
    club_name = r["_club_name"]
    club_uri  = make_uri(club_name)
    if club_uri:
        g.add((club_uri, RDF.type,   EX.Club))
        g.add((club_uri, RDFS.label, Literal(club_name, lang="en")))
        label_to_qid[str(club_uri)] = r["_club_qid"]
        g.add((player_uri, EX.playsFor, club_uri))

    # Position
    if "pos" in r:
        pos_label = r.get("posLabel", {}).get("value", "")
        pos_uri   = make_uri(pos_label)
        if pos_uri and pos_label:
            g.add((pos_uri,    RDF.type,   EX.Position))
            g.add((pos_uri,    RDFS.label, Literal(pos_label, lang="en")))
            label_to_qid[str(pos_uri)] = qid_from_wd_uri(r["pos"]["value"])
            g.add((player_uri, EX.hasPosition, pos_uri))

    # Nationality
    if "country" in r:
        country_label = r.get("countryLabel", {}).get("value", "")
        country_uri   = make_uri(country_label)
        if country_uri and country_label:
            g.add((country_uri, RDF.type,   EX.Country))
            g.add((country_uri, RDFS.label, Literal(country_label, lang="en")))
            label_to_qid[str(country_uri)] = qid_from_wd_uri(r["country"]["value"])
            g.add((player_uri, EX.hasNationality, country_uri))

    # Birth date
    dob = to_xsd_date(r.get("dob", {}).get("value"))
    if dob:
        g.add((player_uri, EX.birthDate, Literal(dob, datatype=XSD.date)))

    # Height
    if "height" in r:
        try:
            h = float(r["height"]["value"])
            if not math.isnan(h) and 1.0 < h < 2.5:
                g.add((player_uri, EX.heightM, Literal(round(h, 2), datatype=XSD.decimal)))
        except:
            pass

    # Birth place
    if "birthPlace" in r:
        bp_label = r.get("birthPlaceLabel", {}).get("value", "")
        bp_uri   = make_uri(bp_label)
        if bp_uri and bp_label:
            g.add((bp_uri,     RDF.type,   EX.Place))
            g.add((bp_uri,     RDFS.label, Literal(bp_label, lang="en")))
            label_to_qid[str(bp_uri)] = qid_from_wd_uri(r["birthPlace"]["value"])
            g.add((player_uri, EX.birthPlace, bp_uri))

g.serialize(destination="private_kb.ttl", format="turtle")
print("Saved: private_kb.ttl")
print("Triples:", len(g))

Saved: private_kb.ttl
Triples: 45850


In [5]:
# Stats
subjects = set(g.subjects())
objects  = set(o for o in g.objects() if isinstance(o, URIRef))
entities = subjects | objects
print("Entities   :", len(entities))
print("Relations :", len(set(g.predicates())))

types = {}
for s,p,o in g:
    if str(p).endswith("type"):
        t = str(o).split("/")[-1]
        types[t] = types.get(t, 0) + 1
for k,v in sorted(types.items(), key=lambda x:-x[1]):
    print(f"  {k:<20} : {v}")

Entities   : 8324
Relations : 8
  FootballPlayer       : 5931
  Place                : 2271
  Country              : 102
  Position             : 26
  Club                 : 6


## Step 2: Entity Linking with Open Knowledge Bases

In [6]:
# IGNOREE
from rdflib.namespace import OWL

WD = Namespace("http://www.wikidata.org/entity/")
g.bind("wd",  WD)
g.bind("owl", OWL)

def as_wd_entity(ex_uri: URIRef):
    """
    Convert ex:player/Q123 → wd:Q123
    Work for player/, club/, position/, country/, place/
    """
    s = str(ex_uri)
    # We try to find the QID in the last segment
    last = s.rsplit("/", 1)[-1]
    if last.startswith("Q") and last[1:].isdigit():
        return WD[last]
    return None

linked_count = 0
all_uris = set(g.subjects()) | set(o for o in g.objects() if isinstance(o, URIRef))

for uri in all_uris:
    wd_uri = as_wd_entity(uri)
    if wd_uri is not None:
        g.add((uri, OWL.sameAs, wd_uri))
        linked_count += 1

print(f"owl:sameAs ajoutés : {linked_count}")
print(f"Triples total      : {len(g)}")

owl:sameAs ajoutés : 17
Triples total      : 45867


In [7]:
from rdflib.namespace import OWL

WD = Namespace("http://www.wikidata.org/entity/")
g.bind("wd",  WD)
g.bind("owl", OWL)

# Now we use label_to_qid built in Step 1
# because QIDs are no longer in the URIs
linked_count = 0
mapping_rows = []

for ex_uri_str, qid in label_to_qid.items():
    if not qid:
        continue
    ex_uri = URIRef(ex_uri_str)
    wd_uri = WD[qid]
    g.add((ex_uri, OWL.sameAs, wd_uri))
    linked_count += 1
    mapping_rows.append({
        "Private Entity": ex_uri_str,
        "External URI":   str(wd_uri),
        "Confidence":     1.0
    })

print(f"owl:sameAs added : {linked_count}")
print(f"Total triples    : {len(g)}")

owl:sameAs added : 8319
Total triples    : 54169


In [8]:
# Vérification :  show 10 examples
count = 0
for s, p, o in g.triples((None, OWL.sameAs, None)):
    print(s, "owl:sameAs", o)
    count += 1
    if count >= 10:
        break

http://example.org/Q12263850 owl:sameAs http://www.wikidata.org/entity/Q12263850
http://example.org/Q494305 owl:sameAs http://www.wikidata.org/entity/Q494305
http://example.org/Q785 owl:sameAs http://www.wikidata.org/entity/Q785
http://example.org/Q12393893 owl:sameAs http://www.wikidata.org/entity/Q12393893
http://example.org/Q2291466 owl:sameAs http://www.wikidata.org/entity/Q2291466
http://example.org/Q32172600 owl:sameAs http://www.wikidata.org/entity/Q32172600
http://example.org/Q95380 owl:sameAs http://www.wikidata.org/entity/Q95380
http://example.org/Q118955322 owl:sameAs http://www.wikidata.org/entity/Q118955322
http://example.org/Q138000464 owl:sameAs http://www.wikidata.org/entity/Q138000464
http://example.org/Q2057625 owl:sameAs http://www.wikidata.org/entity/Q2057625


In [9]:
g.serialize(destination="private_kb_linked.ttl", format="turtle")
print("Saved: private_kb_linked.ttl")
print("Triples:", len(g))

Saved: private_kb_linked.ttl
Triples: 54169


In [10]:
# IGNORE
# Table for mapping
import pandas as pd

rows = []
for s, p, o in g.triples((None, OWL.sameAs, None)):
    rows.append({
        "Private Entity":  str(s),
        "External URI":    str(o),
        "Confidence":      1.0
    })

df_mapping = pd.DataFrame(rows)
df_mapping.to_csv("mapping_table.csv", index=False)
print(df_mapping.head(10))
print(f"\nTotal entities linked : {len(df_mapping)}")

                  Private Entity                               External URI  \
0   http://example.org/Q12263850   http://www.wikidata.org/entity/Q12263850   
1     http://example.org/Q494305     http://www.wikidata.org/entity/Q494305   
2        http://example.org/Q785        http://www.wikidata.org/entity/Q785   
3   http://example.org/Q12393893   http://www.wikidata.org/entity/Q12393893   
4    http://example.org/Q2291466    http://www.wikidata.org/entity/Q2291466   
5   http://example.org/Q32172600   http://www.wikidata.org/entity/Q32172600   
6      http://example.org/Q95380      http://www.wikidata.org/entity/Q95380   
7  http://example.org/Q118955322  http://www.wikidata.org/entity/Q118955322   
8  http://example.org/Q138000464  http://www.wikidata.org/entity/Q138000464   
9    http://example.org/Q2057625    http://www.wikidata.org/entity/Q2057625   

   Confidence  
0         1.0  
1         1.0  
2         1.0  
3         1.0  
4         1.0  
5         1.0  
6         1.0  
7 

In [11]:
df_mapping = pd.DataFrame(mapping_rows)
df_mapping.to_csv("mapping_table.csv", index=False)
print(df_mapping.head(10))
print(f"\nTotal entities linked : {len(df_mapping)}")

                        Private Entity  \
0       http://example.org/JimmyRimmer   
1  http://example.org/ManchesterUnited   
2        http://example.org/Goalkeeper   
3     http://example.org/UnitedKingdom   
4         http://example.org/Southport   
5     http://example.org/JovanKirovski   
6           http://example.org/Forward   
7      http://example.org/UnitedStates   
8         http://example.org/Escondido   
9       http://example.org/ShaunGoater   

                              External URI  Confidence  
0  http://www.wikidata.org/entity/Q1370782         1.0  
1    http://www.wikidata.org/entity/Q18656         1.0  
2   http://www.wikidata.org/entity/Q201330         1.0  
3      http://www.wikidata.org/entity/Q145         1.0  
4   http://www.wikidata.org/entity/Q868647         1.0  
5  http://www.wikidata.org/entity/Q1382753         1.0  
6   http://www.wikidata.org/entity/Q280658         1.0  
7       http://www.wikidata.org/entity/Q30         1.0  
8   http://www.wikidata.

## Step 3: Predicate Alignment Using SPARQL

In [12]:
from rdflib.namespace import OWL

WDT = Namespace("http://www.wikidata.org/prop/direct/")
g.bind("wdt", WDT)

# Alignment: private predicate -> equivalent Wikidata property
predicate_alignment = {
    EX.playsFor:       WDT.P54,    # member of sports team
    EX.hasPosition:    WDT.P413,   # position played
    EX.hasNationality: WDT.P27,    # country of citizenship
    EX.birthPlace:     WDT.P19,    # place of birth
    EX.birthDate:      WDT.P569,   # date of birth
    EX.heightM:        WDT.P2048,  # height
}

added = 0
for p_ex, p_wdt in predicate_alignment.items():
    g.add((p_ex, OWL.equivalentProperty, p_wdt))
    added += 1

print(f"Alignements added : {added}")
print(f"Triples total       : {len(g)}")

Alignements added : 6
Triples total       : 54175


In [13]:
# Verification
for p_ex, p_wdt in predicate_alignment.items():
    ok = (p_ex, OWL.equivalentProperty, p_wdt) in g
    print(f"{str(p_ex).split('/')[-1]:<20} ≡  {str(p_wdt).split('/')[-1]}  ->  {'OK' if ok else 'MISSING'}")

playsFor             ≡  P54  ->  OK
hasPosition          ≡  P413  ->  OK
hasNationality       ≡  P27  ->  OK
birthPlace           ≡  P19  ->  OK
birthDate            ≡  P569  ->  OK
heightM              ≡  P2048  ->  OK


In [14]:
g.serialize(destination="private_kb_linked_aligned.ttl", format="turtle")
print("Saved: private_kb_linked_aligned.ttl")
print("Triples:", len(g))

Saved: private_kb_linked_aligned.ttl
Triples: 54175


## Step 4: KB Expansion via SPARQL

In [15]:
import requests, time
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
WD  = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

# Load the aligned KB
g_exp = Graph()
g_exp.parse("private_kb_linked_aligned.ttl", format="turtle")
print(f"Starting KB : {len(g_exp)} triples")

# Extract all anchor QIDs from owl:sameAs links
anchor_qids = set()
for s, p, o in g_exp.triples((None, OWL.sameAs, None)):
    qid = str(o).split("/")[-1]
    if qid.startswith("Q") and qid[1:].isdigit():
        anchor_qids.add(qid)

print(f"Anchor entities found : {len(anchor_qids)}")

Starting KB : 54175 triples
Anchor entities found : 8319


In [16]:
# Allowed Wikidata predicates (avoids heavy/useless literals)
ALLOWED_PIDS = [
    "P54",   # member of sports team
    "P27",   # country of citizenship
    "P413",  # position played
    "P569",  # date of birth
    "P19",   # place of birth
    "P20",   # place of death
    "P106",  # occupation
    "P166",  # award received
    "P17",   # country
    "P131",  # located in administrative entity
    "P31",   # instance of
    "P641",  # sport
    "P118",  # league
    "P115",  # home venue / stadium
    "P571",  # inception (founding date)
    "P286",  # head coach
    "P1532", # country for sport
    "P2048", # height
    "P1351",   # number of goals scored
    "P6509",   # total goals in career
    "P2276",   # UEFA club competition appearances
    "P552",    # handedness (left/right foot)
    "P741",    # playing hand
    "P1532",   # country for sport
    "P97",     # noble title (for captaincy)
    "P69",     # educated at
    "P101",    # field of work
    "P1344",   # participant in (World Cup, Euro, etc.)
    "P555",    # shirt number
    "P6087",   # coach of sports team
    "P2522",   # victory
    "P1346",   # winner of
    "P1923",   # participating team
    "P585",    # point in time
    "P580",    # start time
    "P582",    # end time
]
FILTER_PIDS = " || ".join([f"?p = wdt:{pid}" for pid in ALLOWED_PIDS])

def add_triple(g, s_uri, p_uri, row_o):
    """Add a triple to the graph from a SPARQL result row"""
    o_val  = row_o.get("value", "")
    o_type = row_o.get("type", "")
    if not o_val:
        return
    if o_type == "uri":
        g.add((s_uri, p_uri, URIRef(o_val)))
    elif o_type == "literal":
        lang  = row_o.get("xml:lang", "")
        dtype = row_o.get("datatype", "")
        if lang:
            g.add((s_uri, p_uri, Literal(o_val, lang=lang)))
        elif dtype:
            g.add((s_uri, p_uri, Literal(o_val, datatype=URIRef(dtype))))
        else:
            g.add((s_uri, p_uri, Literal(o_val)))

def batch_one_hop(qids: list, limit=500) -> list:
    """1-hop expansion on a batch of QIDs in a single SPARQL query"""
    values = " ".join([f"wd:{q}" for q in qids])
    query = f"""
    PREFIX wd:  <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>

    SELECT ?s ?p ?o WHERE {{
      VALUES ?s {{ {values} }}
      ?s ?p ?o .
      FILTER({FILTER_PIDS})
      FILTER(!isLiteral(?o) || LANG(?o) = "en" || LANG(?o) = "")
    }}
    LIMIT {limit}
    """
    try:
        resp = requests.get(
            SPARQL_ENDPOINT,
            params={"query": query, "format": "json"},
            headers={"User-Agent": "FootballKB/1.0"},
            timeout=30
        )
        if resp.status_code == 200:
            return resp.json()["results"]["bindings"]
        return []
    except:
        return []

def two_hop_batch(qids: list, limit=300) -> list:
    """2-hop expansion: player -> club -> club info. Fetches properties of clubs linked to a batch of players"""
    values = " ".join([f"wd:{q}" for q in qids])
    query = f"""
    PREFIX wd:  <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>

    SELECT ?club ?p ?o WHERE {{
      VALUES ?player {{ {values} }}
      ?player wdt:P54 ?club .
      ?club ?p ?o .
      FILTER({FILTER_PIDS})
      FILTER(!isLiteral(?o) || LANG(?o) = "en" || LANG(?o) = "")
    }}
    LIMIT {limit}
    """
    try:
        resp = requests.get(
            SPARQL_ENDPOINT,
            params={"query": query, "format": "json"},
            headers={"User-Agent": "FootballKB/1.0"},
            timeout=30
        )
        if resp.status_code == 200:
            return resp.json()["results"]["bindings"]
        return []
    except:
        return []

def predicate_expansion(pid: str, limit=5000) -> list:
    """ Predicate-controlled expansion:fetch all footballers from Wikidata that have a given property"""
    query = f"""
    PREFIX wd:  <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>

    SELECT ?s ?o WHERE {{
      ?s wdt:P31  wd:Q5 .
      ?s wdt:P106 wd:Q937857 .
      ?s wdt:{pid} ?o .
    }}
    LIMIT {limit}
    """
    try:
        resp = requests.get(
            SPARQL_ENDPOINT,
            params={"query": query, "format": "json"},
            headers={"User-Agent": "FootballKB/1.0"},
            timeout=30
        )
        if resp.status_code == 200:
            return resp.json()["results"]["bindings"]
        return []
    except:
        return []

In [17]:
# Phase 1: 1-hop expansion in batches of 50 QIDs
# We only expand players (not places/countries) to keep it manageable
print("=== Phase 1: 1-hop batch expansion (players only) ===")

# Keep only player QIDs for 1-hop (reduces from 7000 to 5000)
player_qids = [
    qid for qid in anchor_qids
    if any(
        str(s).endswith(qid)
        for s in g_exp.subjects(RDF.type, URIRef("http://example.org/FootballPlayer"))
    )
]
print(f"Player QIDs to expand : {len(player_qids)}")

# Process in batches of 50
BATCH_SIZE = 50
batches = [player_qids[i:i+BATCH_SIZE] for i in range(0, len(player_qids), BATCH_SIZE)]
print(f"Number of batches     : {len(batches)}")

for i, batch in enumerate(batches):
    rows = batch_one_hop(batch, limit=500)
    for row in rows:
        s_uri = URIRef(row["s"]["value"])
        p_uri = URIRef(row["p"]["value"])
        add_triple(g_exp, s_uri, p_uri, row["o"])

    if (i+1) % 10 == 0 or (i+1) == len(batches):
        print(f"  Batch [{i+1}/{len(batches)}] — triples : {len(g_exp):,}")

    time.sleep(0.8)  # Respect Wikidata rate limit

print(f"\nAfter phase 1 : {len(g_exp):,} triples")

=== Phase 1: 1-hop batch expansion (players only) ===
Player QIDs to expand : 5
Number of batches     : 1
  Batch [1/1] — triples : 54,223

After phase 1 : 54,223 triples


In [18]:
# Phase 2: 2-hop expansion (player -> club -> club info)
print("=== Phase 2: 2-hop expansion (player -> club) ===")

# Use only first 200 players for 2-hop to save time
sample_players = player_qids[:200]
batches_2hop   = [sample_players[i:i+BATCH_SIZE] for i in range(0, len(sample_players), BATCH_SIZE)]

for i, batch in enumerate(batches_2hop):
    rows = two_hop_batch(batch, limit=300)
    for row in rows:
        club_uri = URIRef(row["club"]["value"])
        p_uri    = URIRef(row["p"]["value"])
        add_triple(g_exp, club_uri, p_uri, row["o"])

    if (i+1) % 5 == 0 or (i+1) == len(batches_2hop):
        print(f"  Batch [{i+1}/{len(batches_2hop)}] — triples : {len(g_exp):,}")

    time.sleep(0.8)

print(f"\nAfter phase 2 : {len(g_exp):,} triples")

=== Phase 2: 2-hop expansion (player -> club) ===
  Batch [1/1] — triples : 54,353

After phase 2 : 54,353 triples


In [19]:
# Phase 3: Predicate-controlled expansion
# Fetches all footballers on Wikidata that have these properties
print("=== Phase 3: Predicate-controlled expansion ===")

controlled_preds = [
    ("P54",   3000),  # club membership
    ("P27",   3000),  # nationality
    ("P166",  2000),  # awards received
    ("P413",  2000),  # position played
    ("P19",   2000),  # place of birth
    ("P569",  2000),  # date of birth
    ("P106",  2000),  # occupation
    ("P20",   2000),  # place of death
    ("P17",   2000),  # country
    ("P131",  2000),  # located in administrative entity
    ("P31",   2000),  # instance of
    ("P641",  2000),  # sport
    ("P118",  2000),  # league
    ("P115",  2000),  # home venue
    ("P571",  2000),  # founding date
    ("P286",  2000),  # head coach
    ("P1532", 2000),  # country for sport
    ("P2048", 2000),  # height
    ("P21",   2000),  # sex or gender
    ("P40",   1000),  # child
    ("P22",   1000),  # father
    ("P25",   1000),  # mother
    ("P26",   1000),  # spouse
    ("P108",  2000),  # employer
    ("P award",1000), # notable work
    ("P1344", 2000),  # participant in
    ("P medal",1000), # participant in
    ("P460",  1000),  # said to be the same as
    ("P641",  1000),  # sport
    ("P2046", 1000),  # area
    ("P276",  1000),  # location
    ("P159",  1000),  # headquarters
    ("P856",  1000),  # official website
    ("P18",   1000),  # image
    ("P41",   1000),  # flag image
    ("P94",   1000),  # coat of arms
    ("P242",  1000),  # locator map
    ("P281",  1000),  # postal code
    ("P625",  1000),  # coordinate location
    ("P30",   1000),  # continent
    ("P36",   1000),  # capital
    ("P37",   1000),  # official language
    ("P38",   1000),  # currency
    ("P47",   1000),  # shares border with
    ("P150",  1000),  # contains administrative territorial entity
    ("P190",  1000),  # twinned administrative body
    ("P194",  1000),  # legislative body
    ("P6",    1000),  # head of government
    ("P35",   1000),  # head of state
    ("P530",  1000),  # diplomatic relation
    ("P1566", 1000),  # GeoNames ID
    ("P910",  1000),  # topic's main category
]

for pid, limit in controlled_preds:
    rows  = predicate_expansion(pid, limit=limit)
    p_uri = URIRef(f"http://www.wikidata.org/prop/direct/{pid}")
    for row in rows:
        s_uri = URIRef(row["s"]["value"])
        add_triple(g_exp, s_uri, p_uri, row["o"])
    print(f"  wdt:{pid} -> +{len(rows):,} rows  (total triples: {len(g_exp):,})")
    time.sleep(1.5)

print(f"\nAfter phase 3 : {len(g_exp):,} triples")

=== Phase 3: Predicate-controlled expansion ===
  wdt:P54 -> +3,000 rows  (total triples: 57,353)
  wdt:P27 -> +3,000 rows  (total triples: 60,353)
  wdt:P166 -> +2,000 rows  (total triples: 62,353)
  wdt:P413 -> +0 rows  (total triples: 62,353)
  wdt:P19 -> +2,000 rows  (total triples: 64,353)
  wdt:P569 -> +2,000 rows  (total triples: 66,353)
  wdt:P106 -> +2,000 rows  (total triples: 68,353)
  wdt:P20 -> +2,000 rows  (total triples: 70,353)
  wdt:P17 -> +0 rows  (total triples: 70,353)
  wdt:P131 -> +0 rows  (total triples: 70,353)
  wdt:P31 -> +0 rows  (total triples: 70,353)
  wdt:P641 -> +2,000 rows  (total triples: 72,353)
  wdt:P118 -> +2,000 rows  (total triples: 74,353)
  wdt:P115 -> +0 rows  (total triples: 74,353)
  wdt:P571 -> +0 rows  (total triples: 74,353)
  wdt:P286 -> +59 rows  (total triples: 74,412)
  wdt:P1532 -> +2,000 rows  (total triples: 76,412)
  wdt:P2048 -> +2,000 rows  (total triples: 78,412)
  wdt:P21 -> +2,000 rows  (total triples: 80,412)
  wdt:P40 -> +1

http://www.wikidata.org/prop/direct/P award does not look like a valid URI, trying to serialize this will break.


  wdt:P award -> +0 rows  (total triples: 85,228)
  wdt:P1344 -> +2,000 rows  (total triples: 87,228)


http://www.wikidata.org/prop/direct/P medal does not look like a valid URI, trying to serialize this will break.


  wdt:P medal -> +0 rows  (total triples: 87,228)
  wdt:P460 -> +40 rows  (total triples: 87,268)
  wdt:P641 -> +1,000 rows  (total triples: 87,459)
  wdt:P2046 -> +0 rows  (total triples: 87,459)
  wdt:P276 -> +0 rows  (total triples: 87,459)
  wdt:P159 -> +1 rows  (total triples: 87,460)
  wdt:P856 -> +1,000 rows  (total triples: 88,460)
  wdt:P18 -> +1,000 rows  (total triples: 89,460)
  wdt:P41 -> +1 rows  (total triples: 89,461)
  wdt:P94 -> +8 rows  (total triples: 89,469)
  wdt:P242 -> +0 rows  (total triples: 89,469)
  wdt:P281 -> +0 rows  (total triples: 89,469)
  wdt:P625 -> +0 rows  (total triples: 89,469)
  wdt:P30 -> +0 rows  (total triples: 89,469)
  wdt:P36 -> +0 rows  (total triples: 89,469)
  wdt:P37 -> +1 rows  (total triples: 89,470)
  wdt:P38 -> +0 rows  (total triples: 89,470)
  wdt:P47 -> +0 rows  (total triples: 89,470)
  wdt:P150 -> +0 rows  (total triples: 89,470)
  wdt:P190 -> +0 rows  (total triples: 89,470)
  wdt:P194 -> +0 rows  (total triples: 89,470)
  wd

In [20]:
# Final stats and save
entities  = set(str(s) for s,p,o in g_exp if isinstance(s, URIRef))
entities |= set(str(o) for s,p,o in g_exp if isinstance(o, URIRef))
relations = set(str(p) for s,p,o in g_exp)

print(f"\n{'─'*40}")
print(f"  FINAL STATISTICS")
print(f"{'─'*40}")
print(f"  Triples   : {len(g_exp):>10,}")
print(f"  Entities  : {len(entities):>10,}")
print(f"  Relations : {len(relations):>10,}")
print(f"\nThreshold check (lab requirements):")
print(f"  50k-200k triples  : {'OK' if 50000  <= len(g_exp)     <= 200000 else 'OUT OF RANGE'} ({len(g_exp):,})")
print(f"  5k-30k entities   : {'OK' if 5000   <= len(entities)  <= 30000  else 'OUT OF RANGE'} ({len(entities):,})")
print(f"  50-200 relations  : {'OK' if 50     <= len(relations) <= 200    else 'OUT OF RANGE'} ({len(relations):,})")

# Save in both formats
g_exp.serialize("expanded_kb.ttl", format="turtle")
g_exp.serialize("expanded_kb.nt",  format="nt")
print(f"\nSaved: expanded_kb.ttl")
print(f"Saved: expanded_kb.nt  (for KGE)")


────────────────────────────────────────
  FINAL STATISTICS
────────────────────────────────────────
  Triples   :     89,657
  Entities  :     36,007
  Relations :         43

Threshold check (lab requirements):
  50k-200k triples  : OK (89,657)
  5k-30k entities   : OUT OF RANGE (36,007)
  50-200 relations  : OUT OF RANGE (43)


C:\Users\marce\anaconda3\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(



Saved: expanded_kb.ttl
Saved: expanded_kb.nt  (for KGE)


## Ontology 

In [21]:
# Ontology — add at the end of TD4, after Step 4

ontology = """@prefix ex:  <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

# === Classes ===

ex:Person a owl:Class ;
    rdfs:label "Person"@en .

ex:FootballPlayer a owl:Class ;
    rdfs:subClassOf ex:Person ;
    rdfs:label "Football Player"@en .

ex:Club a owl:Class ;
    rdfs:label "Football Club"@en .

ex:Country a owl:Class ;
    rdfs:label "Country"@en .

ex:Place a owl:Class ;
    rdfs:label "Place"@en .

ex:Position a owl:Class ;
    rdfs:label "Playing Position"@en .

# === Object Properties ===

ex:playsFor a owl:ObjectProperty ;
    rdfs:domain ex:FootballPlayer ;
    rdfs:range  ex:Club ;
    rdfs:label  "plays for"@en .

ex:hasNationality a owl:ObjectProperty ;
    rdfs:domain ex:FootballPlayer ;
    rdfs:range  ex:Country ;
    rdfs:label  "has nationality"@en .

ex:hasPosition a owl:ObjectProperty ;
    rdfs:domain ex:FootballPlayer ;
    rdfs:range  ex:Position ;
    rdfs:label  "plays position"@en .

ex:birthPlace a owl:ObjectProperty ;
    rdfs:domain ex:FootballPlayer ;
    rdfs:range  ex:Place ;
    rdfs:label  "birth place"@en .

# === Datatype Properties ===

ex:birthDate a owl:DatatypeProperty ;
    rdfs:domain ex:FootballPlayer ;
    rdfs:range  xsd:date ;
    rdfs:label  "birth date"@en .

ex:heightM a owl:DatatypeProperty ;
    rdfs:domain ex:FootballPlayer ;
    rdfs:range  xsd:decimal ;
    rdfs:label  "height in meters"@en .
"""

with open("ontology.ttl", "w") as f:
    f.write(ontology)

print("Saved: ontology.ttl")

Saved: ontology.ttl
